# SegFormer B2 for Clothes Segmentation (`mattmdjaga/segformer_b2_clothes`)
## A Complete Technical Reference

---

## 1. High-Level Overview

### 1.1 Purpose and Problem Statement

**`mattmdjaga/segformer_b2_clothes`** is a fine-tuned instance of the **SegFormer-B2** architecture for **human clothing/body-part semantic segmentation**. Given an input RGB image of a person, the model produces a pixel-wise classification map assigning each pixel to one of **18 semantic classes**:

| Label ID | Class | Label ID | Class |
|----------|-------|----------|-------|
| 0 | Background | 9 | Left-shoe |
| 1 | Hat | 10 | Right-shoe |
| 2 | Hair | 11 | Face |
| 3 | Sunglasses | 12 | Left-leg |
| 4 | Upper-clothes | 13 | Right-leg |
| 5 | Skirt | 14 | Left-arm |
| 6 | Pants | 15 | Right-arm |
| 7 | Dress | 16 | Bag |
| 8 | Belt | 17 | Scarf |

### 1.2 Why SegFormer Was Introduced

Prior to SegFormer (Xie et al., NeurIPS 2021), semantic segmentation was dominated by:

1. **CNN-based methods** (FCN, DeepLab, PSPNet): Strong local feature extraction but limited global context without dilated convolutions or large receptive fields.
2. **Early Vision Transformers** (SETR, ViT-based): Good global context but computationally expensive (quadratic attention), required positional encodings (limiting resolution flexibility), and used heavy decoders.

SegFormer was introduced to unify the **efficiency of CNNs** with the **global modeling capacity of Transformers** while eliminating key limitations:

- **No positional encoding** → resolution-agnostic at inference
- **Hierarchical multi-scale features** → like CNNs, captures both local and global patterns
- **Efficient self-attention** → linear complexity via sequence reduction
- **Lightweight MLP decoder** → minimal overhead, leverages powerful encoder representations

### 1.3 Comparison with Previous Approaches

| Aspect | FCN/DeepLab | SETR (ViT) | SegFormer |
|--------|-------------|------------|------------|
| Backbone | CNN (ResNet) | ViT (single-scale) | Hierarchical Transformer (MiT) |
| Receptive field | Limited/dilated | Global (quadratic cost) | Global (linear cost) |
| Positional encoding | N/A | Fixed/learned | None (overlap patch embed) |
| Multi-scale features | Via FPN/ASPP | No (single resolution) | Native (4 stages) |
| Decoder complexity | Heavy (ASPP, FPN) | Heavy (PUP, MLA) | Lightweight All-MLP |
| Resolution flexibility | Fixed or with tricks | Fixed (interpolate PE) | Fully flexible |
| Params (B2 variant) | ~60M (ResNet-101) | ~300M (ViT-Large) | ~27.4M |
| Inference speed | Fast | Slow | Fast |

### 1.4 Strengths

1. **Efficiency**: Linear attention complexity via Sequence Reduction (SR) makes it practical for high-resolution inputs
2. **No positional encoding**: Overlapping patch embeddings implicitly encode position; model generalizes to arbitrary resolutions at test time
3. **Multi-scale representations**: Hierarchical 4-stage encoder naturally captures features at multiple scales
4. **Simple decoder**: All-MLP decoder avoids complex upsampling, yet achieves SOTA results
5. **Scalable family**: B0–B5 variants offer accuracy-efficiency trade-offs
6. **Strong fine-tuning**: Pre-trained on ImageNet-1K, transfers well to downstream tasks like clothes parsing

### 1.5 Weaknesses

1. **Sequence Reduction trade-off**: Aggressive spatial reduction in attention may lose fine-grained spatial details
2. **Limited to semantic segmentation**: No instance or panoptic segmentation without additional heads
3. **Patch-based tokenization**: Boundary artifacts possible at patch boundaries
4. **Training data dependency**: The clothes model quality depends on the ATR/CFPD fine-tuning dataset quality and diversity
5. **No explicit edge modeling**: Unlike methods with CRF post-processing, boundaries may be less crisp

## 2. Mathematical Foundations

### 2.1 Standard Multi-Head Self-Attention (MHSA)

#### 2.1.1 Setup and Notation

Given an input sequence $$\mathbf{X} \in \mathbb{R}^{N \times C}$$ where:
- $$N = H \times W$$ is the number of spatial tokens (flattened 2D feature map)
- $$C$$ is the embedding dimension
- $$H, W$$ are height and width of the feature map

Standard self-attention computes:

$$\mathbf{Q} = \mathbf{X}\mathbf{W}_Q, \quad \mathbf{K} = \mathbf{X}\mathbf{W}_K, \quad \mathbf{V} = \mathbf{X}\mathbf{W}_V$$

where $$\mathbf{W}_Q, \mathbf{W}_K, \mathbf{W}_V \in \mathbb{R}^{C \times d_h}$$ and $$d_h = C / N_h$$ (head dimension, $$N_h$$ = number of heads).

#### 2.1.2 Attention Computation

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_h}}\right)\mathbf{V}$$

**Derivation of the scaling factor $$\sqrt{d_h}$$:**

Assume $$q_i, k_j \sim \mathcal{N}(0, 1)$$ i.i.d. Then:
$$\text{Var}(\mathbf{q}^T\mathbf{k}) = \text{Var}\left(\sum_{i=1}^{d_h} q_i k_i\right) = \sum_{i=1}^{d_h} \text{Var}(q_i)\text{Var}(k_i) = d_h$$

Without scaling, dot products grow with $$d_h$$, pushing softmax into saturated regions with vanishing gradients. Dividing by $$\sqrt{d_h}$$ normalizes variance to 1.

#### 2.1.3 Multi-Head Formulation

$$\text{MHSA}(\mathbf{X}) = \text{Concat}(\text{head}_1, \ldots, \text{head}_{N_h})\mathbf{W}_O$$

where $$\text{head}_i = \text{Attention}(\mathbf{X}\mathbf{W}_{Q_i}, \mathbf{X}\mathbf{W}_{K_i}, \mathbf{X}\mathbf{W}_{V_i})$$ and $$\mathbf{W}_O \in \mathbb{R}^{C \times C}$$.

**Complexity**: $$\mathcal{O}(N^2 \cdot C)$$ — quadratic in sequence length. For a $$512 \times 512$$ image with patch size 4, $$N = 128 \times 128 = 16384$$, making standard attention prohibitive.

---

### 2.2 Efficient Self-Attention with Sequence Reduction (SR)

SegFormer's key innovation: **reduce the spatial dimension of K and V before computing attention**.

#### 2.2.1 Sequence Reduction Operation

Given $$\mathbf{K} \in \mathbb{R}^{N \times C}$$, reshape to $$\mathbf{K} \in \mathbb{R}^{\frac{N}{R^2} \times (C \cdot R^2)}$$ where $$R$$ is the reduction ratio.

Then apply a linear projection:

$$\hat{\mathbf{K}} = \text{Reshape}(\mathbf{K}, [N/R^2, C \cdot R^2]) \cdot \mathbf{W}_{SR} + \mathbf{b}_{SR}$$

where $$\mathbf{W}_{SR} \in \mathbb{R}^{(C \cdot R^2) \times C}$$, yielding $$\hat{\mathbf{K}} \in \mathbb{R}^{N/R^2 \times C}$$.

Same operation applied to $$\mathbf{V}$$:
$$\hat{\mathbf{V}} = \text{Reshape}(\mathbf{V}, [N/R^2, C \cdot R^2]) \cdot \mathbf{W}_{SR}^V + \mathbf{b}_{SR}^V$$

Followed by Layer Normalization:
$$\hat{\mathbf{K}} = \text{LayerNorm}(\hat{\mathbf{K}}), \quad \hat{\mathbf{V}} = \text{LayerNorm}(\hat{\mathbf{V}})$$

#### 2.2.2 Efficient Attention Formula

$$\text{EfficientAttn}(\mathbf{Q}, \hat{\mathbf{K}}, \hat{\mathbf{V}}) = \text{softmax}\left(\frac{\mathbf{Q}\hat{\mathbf{K}}^T}{\sqrt{d_h}}\right)\hat{\mathbf{V}}$$

where $$\mathbf{Q} \in \mathbb{R}^{N \times d_h}$$, $$\hat{\mathbf{K}} \in \mathbb{R}^{N/R^2 \times d_h}$$.

**Complexity**: $$\mathcal{O}(N^2/R^2 \cdot C)$$ — reduced by factor $$R^2$$.

For SegFormer-B2, reduction ratios per stage: $$R = [8, 4, 2, 1]$$
- Stage 1: $$N = H/4 \times W/4$$, $$R=8$$ → attention over $$N/64$$ tokens
- Stage 4: $$N = H/32 \times W/32$$, $$R=1$$ → full attention (small spatial size)

#### 2.2.3 Intuition

The SR operation is equivalent to a **strided spatial pooling followed by a learned linear combination**. It asks: "Instead of attending to every pixel, attend to spatially-grouped summaries." Early layers (large spatial size) use aggressive reduction; later layers (small spatial size) can afford full attention.

---

### 2.3 Overlapping Patch Embedding

#### 2.3.1 Formulation

Instead of non-overlapping patches (ViT), SegFormer uses overlapping patches via strided convolution:

$$\mathbf{X}_{\text{tokens}} = \text{Conv2d}(\mathbf{X}_{\text{input}}, \text{kernel}=K_p, \text{stride}=S_p, \text{padding}=P_p)$$

For SegFormer:
- **Stage 1**: $$K_p = 7, S_p = 4, P_p = 3$$ → output: $$H/4 \times W/4$$
- **Stages 2–4**: $$K_p = 3, S_p = 2, P_p = 1$$ → each halves spatial dims

#### 2.3.2 Output Dimension Calculation

$$H_{out} = \left\lfloor\frac{H_{in} + 2P_p - K_p}{S_p}\right\rfloor + 1$$

For Stage 1 with $$H_{in} = 512$$:
$$H_{out} = \left\lfloor\frac{512 + 6 - 7}{4}\right\rfloor + 1 = \left\lfloor\frac{511}{4}\right\rfloor + 1 = 127 + 1 = 128$$

#### 2.3.3 Why Overlapping Patches Replace Positional Encoding

Overlapping patches with stride < kernel size create **shared boundary information** between adjacent tokens. Each token's representation inherently encodes its spatial relationship to neighbors through these overlapping receptive fields, providing implicit positional information. This is proven by:
1. Padding in zero-padded convolutions leaks absolute position (Islam et al., 2020)
2. Overlapping regions create translation-equivariant local structure
3. Empirically, removing PE with overlap patches shows no performance degradation

---

### 2.4 Mix-FFN (Feed-Forward Network with Depthwise Convolution)

#### 2.4.1 Standard FFN (Transformer)

$$\text{FFN}(\mathbf{x}) = \text{GELU}(\mathbf{x}\mathbf{W}_1 + \mathbf{b}_1)\mathbf{W}_2 + \mathbf{b}_2$$

#### 2.4.2 Mix-FFN (SegFormer)

$$\text{Mix-FFN}(\mathbf{x}) = \text{GELU}(\text{DWConv}_{3\times3}(\mathbf{x}\mathbf{W}_1 + \mathbf{b}_1))\mathbf{W}_2 + \mathbf{b}_2$$

where $$\text{DWConv}_{3\times3}$$ is a $$3 \times 3$$ depthwise separable convolution applied after reshaping the sequence back to 2D spatial form.

**Steps**:
1. Linear projection: $$\mathbf{x} \in \mathbb{R}^{N \times C} \rightarrow \mathbb{R}^{N \times 4C}$$ (expansion ratio = 4)
2. Reshape to 2D: $$\mathbb{R}^{N \times 4C} \rightarrow \mathbb{R}^{H \times W \times 4C}$$
3. Depthwise Conv: $$3 \times 3$$ conv with groups $$= 4C$$ (each channel convolved independently)
4. GELU activation
5. Linear projection: $$\mathbb{R}^{N \times 4C} \rightarrow \mathbb{R}^{N \times C}$$

#### 2.4.3 GELU Activation

$$\text{GELU}(x) = x \cdot \Phi(x) = x \cdot \frac{1}{2}\left[1 + \text{erf}\left(\frac{x}{\sqrt{2}}\right)\right]$$

Approximation: $$\text{GELU}(x) \approx 0.5x\left(1 + \tanh\left[\sqrt{2/\pi}(x + 0.044715x^3)\right]\right)$$

#### 2.4.4 Why Mix-FFN?

The $$3 \times 3$$ depthwise convolution provides:
1. **Local positional information** (further eliminating need for PE)
2. **Local spatial mixing** complementing the global mixing of attention
3. **Minimal parameter overhead** (depthwise: only $$9 \times C_{\text{hidden}}$$ params vs $$C^2$$ for standard conv)

---

### 2.5 Layer Normalization

$$\text{LayerNorm}(\mathbf{x}) = \frac{\mathbf{x} - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

where:
- $$\mu = \frac{1}{C}\sum_{i=1}^C x_i$$ (mean over feature dimension)
- $$\sigma^2 = \frac{1}{C}\sum_{i=1}^C (x_i - \mu)^2$$ (variance over feature dimension)
- $$\gamma, \beta \in \mathbb{R}^C$$ are learned affine parameters
- $$\epsilon = 10^{-6}$$ for numerical stability

---

### 2.6 Loss Function: Cross-Entropy for Semantic Segmentation

#### 2.6.1 Per-Pixel Cross-Entropy

For a single pixel at position $$(i, j)$$ with ground truth class $$y_{ij} \in \{0, 1, \ldots, K-1\}$$ (K=18 for clothes):

$$\ell_{ij} = -\log\left(\frac{\exp(z_{ij,y_{ij}})}{\sum_{k=0}^{K-1} \exp(z_{ij,k})}\right) = -z_{ij,y_{ij}} + \log\sum_{k=0}^{K-1}\exp(z_{ij,k})$$

where $$z_{ij,k}$$ is the logit for class $$k$$ at pixel $$(i,j)$$.

#### 2.6.2 Full Objective

$$\mathcal{L} = \frac{1}{|\Omega|}\sum_{(i,j) \in \Omega} \ell_{ij}$$

where $$\Omega$$ is the set of all valid (non-ignored) pixels.

#### 2.6.3 Derivation from Maximum Likelihood

Assuming pixels are independent (standard assumption), the likelihood is:
$$P(\mathbf{Y}|\mathbf{X}; \theta) = \prod_{(i,j) \in \Omega} P(y_{ij} | \mathbf{X}; \theta)$$

Negative log-likelihood:
$$-\log P(\mathbf{Y}|\mathbf{X}; \theta) = -\sum_{(i,j)} \log P(y_{ij} | \mathbf{X}; \theta) = \sum_{(i,j)} \ell_{ij}$$

This is exactly cross-entropy when $$P(y_{ij} = k | \mathbf{X}) = \text{softmax}(z_{ij})_k$$.

#### 2.6.4 Gradient w.r.t. Logits

$$\frac{\partial \ell_{ij}}{\partial z_{ij,k}} = \text{softmax}(z_{ij})_k - \mathbb{1}[k = y_{ij}] = p_k - \mathbb{1}[k = y_{ij}]$$

**Intuition**: The gradient pushes the predicted probability toward 1 for the correct class and toward 0 for incorrect classes. Magnitude is proportional to the "error" $$p_k - \text{target}_k$$.

#### 2.6.5 Numerical Stability (Log-Sum-Exp Trick)

$$\log\sum_k \exp(z_k) = m + \log\sum_k \exp(z_k - m), \quad m = \max_k z_k$$

This prevents overflow when logits are large.

## 3. Full Architecture: SegFormer-B2

### 3.1 Architecture Overview (ASCII Diagram)

```
Input Image [B, 3, 512, 512]
 │
 │ Stage 1: Overlap Patch Embed (7×7, stride=4)
 ▼
[B, 64, 128, 128] ────────────────────────────────────────────────┐
 │ 2× Transformer Blocks (SR ratio=8) │
 ▼ │
[B, 64, 128, 128] = F1 │
 │ Stage 2: Overlap Patch Embed (3×3, stride=2) │
 ▼ │
[B, 128, 64, 64] │
 │ 4× Transformer Blocks (SR ratio=4) │
 ▼ │
[B, 128, 64, 64] = F2 │
 │ Stage 3: Overlap Patch Embed (3×3, stride=2) │
 ▼ │
[B, 320, 32, 32] │
 │ 4× Transformer Blocks (SR ratio=2) │
 ▼ │
[B, 320, 32, 32] = F3 │
 │ Stage 4: Overlap Patch Embed (3×3, stride=2) │
 ▼ │
[B, 512, 16, 16] │
 │ 2× Transformer Blocks (SR ratio=1) │
 ▼ │
[B, 512, 16, 16] = F4 │
 │ │
 └─────── ALL-MLP DECODER ──────────────────────────────┘
 │
 F1 ── MLP(64→768) ── Upsample(×1) ──┐
 F2 ── MLP(128→768) ── Upsample(×2) ──┤
 F3 ── MLP(320→768) ── Upsample(×4) ──┤ Concat
 F4 ── MLP(512→768) ── Upsample(×8) ──┘
 │
 [B, 4×768, H/4, W/4] = [B, 3072, 128, 128]
 │
 Linear Fuse: Conv1×1(3072 → 768)
 │
 [B, 768, 128, 128]
 │
 Segmentation Head: Conv1×1(768 → 18)
 │
 [B, 18, 128, 128]
 │
 Upsample ×4 (bilinear)
 │
 ▼
[B, 18, 512, 512] ─── Output logits
```

### 3.2 Encoder: Mix Transformer (MiT-B2)

#### 3.2.1 Stage Configuration

| Stage | Embed Dim ($$C_i$$) | Num Heads ($$N_h$$) | Num Blocks | SR Ratio ($$R_i$$) | FFN Ratio | Spatial Res (512 input) |
|-------|-------------------|-------------------|------------|-------------------|-----------|-------------------------|
| 1 | 64 | 1 | 2 | 8 | 8 | 128 × 128 |
| 2 | 128 | 2 | 4 | 4 | 8 | 64 × 64 |
| 3 | 320 | 5 | 4 | 2 | 4 | 32 × 32 |
| 4 | 512 | 8 | 2 | 1 | 4 | 16 × 16 |

#### 3.2.2 Single Transformer Block (within a stage)

```
Input x [B, N, C]
 │
 ├──── LayerNorm
 │ │
 │ Efficient MHSA (SR)
 │ │
 │ DropPath
 │ │
 +───┘ (Residual)
 │
 ├──── LayerNorm
 │ │
 │ Mix-FFN (Linear → DWConv3x3 → GELU → Linear)
 │ │
 │ DropPath
 │ │
 +───┘ (Residual)
 │
Output [B, N, C]
```

Mathematically:
$$\hat{\mathbf{x}} = \text{EMHSA}(\text{LN}(\mathbf{x})) + \mathbf{x}$$
$$\mathbf{x}_{\text{out}} = \text{Mix-FFN}(\text{LN}(\hat{\mathbf{x}})) + \hat{\mathbf{x}}$$

#### 3.2.3 Tensor Dimensions Through the Encoder (B=1, input 512×512)

```
Input: [1, 3, 512, 512]

--- Stage 1 ---
Patch Embed: Conv2d(3, 64, k=7, s=4, p=3) + LayerNorm
  → [1, 64, 128, 128] → reshape → [1, 16384, 64]
Transformer Block 1:
  LN: [1, 16384, 64]
  Q: [1, 1, 16384, 64]  (1 head, d_h=64)
  K reshape: [1, 16384, 64] → [1, 256, 64*64=4096] → Linear(4096,64) → [1, 256, 64]
  Attention: [1, 1, 16384, 256] (Q×K^T, N×N/R² = 16384×256)
  Output: [1, 16384, 64]
  Mix-FFN: [1, 16384, 64] → Linear → [1, 16384, 512] → DWConv → GELU → Linear → [1, 16384, 64]
Transformer Block 2: same
Output F1: [1, 64, 128, 128]

--- Stage 2 ---
Patch Embed: Conv2d(64, 128, k=3, s=2, p=1) + LayerNorm
  → [1, 128, 64, 64] → reshape → [1, 4096, 128]
4× Transformer Blocks (2 heads, SR=4, FFN hidden=1024)
  K/V reduction: [1, 4096, 128] → [1, 256, 128] (R=4, 4096/16=256)
Output F2: [1, 128, 64, 64]

--- Stage 3 ---
Patch Embed: Conv2d(128, 320, k=3, s=2, p=1) + LayerNorm
  → [1, 320, 32, 32] → reshape → [1, 1024, 320]
4× Transformer Blocks (5 heads, SR=2, FFN hidden=1280)
  K/V reduction: [1, 1024, 320] → [1, 256, 320] (R=2, 1024/4=256)
Output F3: [1, 320, 32, 32]

--- Stage 4 ---
Patch Embed: Conv2d(320, 512, k=3, s=2, p=1) + LayerNorm
  → [1, 512, 16, 16] → reshape → [1, 256, 512]
2× Transformer Blocks (8 heads, SR=1, FFN hidden=2048)
  No reduction (full attention on 256 tokens)
Output F4: [1, 512, 16, 16]
```

### 3.3 Decoder: All-MLP

The decoder is deliberately simple to demonstrate that a powerful hierarchical encoder makes complex decoders unnecessary.

#### 3.3.1 Per-Scale Processing

For each feature $$F_i$$ at scale $$i \in \{1, 2, 3, 4\}$$:

$$\hat{F}_i = \text{Linear}(C_i, C_d)(F_i) + \text{Upsample}_{\text{bilinear}}(\hat{F}_i, \text{size}=H/4 \times W/4)$$

where $$C_d = 768$$ is the decoder embedding dimension for B2.

#### 3.3.2 Fusion

$$F_{\text{fused}} = \text{Linear}(4C_d, C_d)\left(\text{Concat}(\hat{F}_1, \hat{F}_2, \hat{F}_3, \hat{F}_4)\right)$$

#### 3.3.3 Classification

$$\text{logits} = \text{Linear}(C_d, N_{\text{classes}})(\text{Dropout}(F_{\text{fused}}))$$

Final bilinear upsample ×4 to original resolution.

### 3.4 Parameter Count Breakdown (MiT-B2)

| Component | Parameters |
|-----------|------------|
| Stage 1 (Patch Embed + 2 Blocks) | ~0.1M |
| Stage 2 (Patch Embed + 4 Blocks) | ~1.0M |
| Stage 3 (Patch Embed + 4 Blocks) | ~6.1M |
| Stage 4 (Patch Embed + 2 Blocks) | ~7.9M |
| **Encoder Total** | **~24.7M** |
| Decoder (MLP + Fuse + Head) | **~2.7M** |
| **Total** | **~27.4M** |

### 3.5 Data Flow Summary

```
RGB Image → [Multi-Scale Hierarchical Encoding] → 4 feature maps at 1/4, 1/8, 1/16, 1/32
           → [MLP Unification to common dim] → 4 feature maps all at C_d channels
           → [Bilinear Upsample to 1/4 scale] → all at H/4 × W/4
           → [Concatenate + Fuse] → single feature map
           → [Classification Head] → per-pixel class logits
           → [Upsample to full res] → final segmentation
```

In [0]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Tuple, List, Optional

# ============================================================================
# COMPONENT 1: Overlapping Patch Embedding
# ============================================================================
# Purpose: Convert spatial feature maps to sequences of token embeddings
# while preserving local continuity through overlap.

class OverlapPatchEmbedding(nn.Module):
    """
    Overlapping Patch Embedding - replaces both non-overlapping tokenization
    AND positional encoding from standard ViT.
    
    Key insight: By using stride < kernel_size, adjacent patches share
    boundary pixels, implicitly encoding relative position.
    
    Args:
        in_channels: Input feature channels (3 for first stage, C_{i-1} otherwise)
        embed_dim: Output embedding dimension C_i
        patch_size: Kernel size for the convolution
        stride: Stride determines spatial downsampling factor
    """
    def __init__(self, in_channels: int, embed_dim: int, 
                 patch_size: int = 7, stride: int = 4):
        super().__init__()
        # Padding chosen to maintain exact downsampling ratio:
        # output_size = input_size / stride (when pad = (patch_size - 1) // 2)
        padding = patch_size // 2
        
        # Single conv layer performs both spatial downsampling and channel projection
        # This is more parameter-efficient than separate pooling + linear
        self.proj = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size,  # Receptive field per token
            stride=stride,          # Spatial reduction factor
            padding=padding         # Maintains floor(H/stride) output size
        )
        # LayerNorm applied in token (sequence) space
        self.norm = nn.LayerNorm(embed_dim)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, int, int]:
        """
        Args:
            x: [B, C_in, H, W]
        Returns:
            tokens: [B, N, C_out] where N = H_out * W_out
            H_out, W_out: spatial dimensions for reshaping back to 2D
        """
        # x: [B, C_in, H, W] -> [B, C_out, H/stride, W/stride]
        x = self.proj(x)
        B, C, H, W = x.shape
        
        # Flatten spatial dims to sequence: [B, C, H, W] -> [B, H*W, C]
        x = x.flatten(2)    # [B, C, H*W]
        x = x.transpose(1, 2)  # [B, H*W, C] = [B, N, C]
        
        # Normalize in feature dimension (stabilizes training)
        x = self.norm(x)
        
        return x, H, W


# ============================================================================
# COMPONENT 2: Efficient Self-Attention with Sequence Reduction
# ============================================================================

class EfficientSelfAttention(nn.Module):
    """
    Efficient Multi-Head Self-Attention with Sequence Reduction (SR).
    
    Instead of computing full N×N attention (O(N²)), reduces K and V
    from N tokens to N/R² tokens before attention computation.
    
    The reduction is learned (linear projection after spatial reshape),
    not a simple pooling - this preserves more information.
    
    Args:
        embed_dim: Token embedding dimension C
        num_heads: Number of attention heads N_h
        sr_ratio: Spatial reduction ratio R (K,V length becomes N/R²)
    """
    def __init__(self, embed_dim: int, num_heads: int, sr_ratio: int = 1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads  # d_h = C / N_h
        self.scale = self.head_dim ** -0.5  # 1/sqrt(d_h) for attention scaling
        self.sr_ratio = sr_ratio
        
        # Linear projections for Q, K, V
        self.q = nn.Linear(embed_dim, embed_dim)  # W_Q: C -> C
        self.kv = nn.Linear(embed_dim, embed_dim * 2)  # W_K and W_V combined
        self.proj = nn.Linear(embed_dim, embed_dim)  # W_O: output projection
        
        # Sequence Reduction: only when R > 1
        if sr_ratio > 1:
            # Conv2d with kernel=stride=R acts as spatial grouping
            # Groups R×R spatial patches into single tokens
            self.sr = nn.Conv2d(
                embed_dim, embed_dim, 
                kernel_size=sr_ratio, stride=sr_ratio
            )
            self.sr_norm = nn.LayerNorm(embed_dim)
    
    def forward(self, x: torch.Tensor, H: int, W: int) -> torch.Tensor:
        """
        Args:
            x: [B, N, C] input tokens
            H, W: spatial dimensions (N = H*W)
        Returns:
            [B, N, C] - same shape as input (attention doesn't change sequence length)
        """
        B, N, C = x.shape
        
        # Compute Q: [B, N, C] -> [B, N_h, N, d_h]
        q = self.q(x).reshape(B, N, self.num_heads, self.head_dim).permute(0, 2, 1, 3)
        # Shape: [B, N_h, N, d_h]
        
        if self.sr_ratio > 1:
            # Reshape sequence back to 2D for spatial reduction
            # [B, N, C] -> [B, C, H, W]
            x_2d = x.permute(0, 2, 1).reshape(B, C, H, W)
            
            # Apply spatial reduction via strided conv
            # [B, C, H, W] -> [B, C, H/R, W/R]
            x_reduced = self.sr(x_2d)
            
            # Flatten back to sequence: [B, C, H/R, W/R] -> [B, N/R², C]
            x_reduced = x_reduced.reshape(B, C, -1).permute(0, 2, 1)
            
            # Normalize reduced sequence
            x_reduced = self.sr_norm(x_reduced)
            
            # Compute K, V from reduced sequence
            # [B, N/R², C] -> [B, N/R², 2C] -> [B, N/R², 2, N_h, d_h]
            kv = self.kv(x_reduced).reshape(B, -1, 2, self.num_heads, self.head_dim)
        else:
            # No reduction at stage 4 (already small spatial size)
            kv = self.kv(x).reshape(B, -1, 2, self.num_heads, self.head_dim)
        
        # Separate K and V: each [B, N_h, N_kv, d_h] where N_kv = N/R²
        kv = kv.permute(2, 0, 3, 1, 4)  # [2, B, N_h, N_kv, d_h]
        k, v = kv[0], kv[1]
        
        # Scaled dot-product attention
        # [B, N_h, N, d_h] @ [B, N_h, d_h, N_kv] = [B, N_h, N, N_kv]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)  # Normalize along key dimension
        
        # Weighted sum of values
        # [B, N_h, N, N_kv] @ [B, N_h, N_kv, d_h] = [B, N_h, N, d_h]
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        
        # Output projection: [B, N, C] -> [B, N, C]
        x = self.proj(x)
        
        return x


# ============================================================================
# COMPONENT 3: Mix-FFN (Feed-Forward with Depthwise Convolution)
# ============================================================================

class MixFFN(nn.Module):
    """
    Mix Feed-Forward Network: Standard FFN + 3x3 Depthwise Convolution.
    
    The DWConv provides:
    1. Local spatial mixing (complementing global attention)
    2. Implicit positional encoding via zero-padding
    3. Inductive bias for local patterns without full convolution cost
    
    Architecture: Linear(C -> 4C) -> DWConv3x3 -> GELU -> Linear(4C -> C)
    
    Args:
        embed_dim: Input/output dimension C
        ffn_ratio: Expansion ratio (hidden_dim = C * ffn_ratio)
    """
    def __init__(self, embed_dim: int, ffn_ratio: int = 4):
        super().__init__()
        hidden_dim = embed_dim * ffn_ratio
        
        # First linear: expand channels
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        
        # Depthwise conv: spatial mixing with groups = hidden_dim
        # Each channel is convolved independently (1 filter per channel)
        # Total params: hidden_dim * 3 * 3 = 9 * hidden_dim (very lightweight)
        self.dwconv = nn.Conv2d(
            hidden_dim, hidden_dim, 
            kernel_size=3, padding=1, 
            groups=hidden_dim  # Key: depthwise = groups equals channels
        )
        
        # Second linear: compress back
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
    
    def forward(self, x: torch.Tensor, H: int, W: int) -> torch.Tensor:
        """
        Args:
            x: [B, N, C] input tokens
            H, W: spatial dimensions for reshaping to 2D for DWConv
        Returns:
            [B, N, C]
        """
        # Linear expansion: [B, N, C] -> [B, N, 4C]
        x = self.fc1(x)
        
        # Reshape to 2D for depthwise conv: [B, N, 4C] -> [B, 4C, H, W]
        B, N, C_hidden = x.shape
        x = x.transpose(1, 2).reshape(B, C_hidden, H, W)
        
        # Depthwise conv: [B, 4C, H, W] -> [B, 4C, H, W]
        # Provides local spatial mixing + positional info via zero-padding
        x = self.dwconv(x)
        
        # Flatten back: [B, 4C, H, W] -> [B, N, 4C]
        x = x.flatten(2).transpose(1, 2)
        
        # GELU activation (smooth approximation to ReLU)
        x = F.gelu(x)
        
        # Linear compression: [B, N, 4C] -> [B, N, C]
        x = self.fc2(x)
        
        return x


# ============================================================================
# COMPONENT 4: Transformer Block
# ============================================================================

class TransformerBlock(nn.Module):
    """
    Single SegFormer Transformer Block:
    x -> LN -> EfficientAttn -> + residual -> LN -> MixFFN -> + residual
    
    Uses Pre-Norm (LN before attention/FFN) which is more stable for training
    deep transformers than Post-Norm.
    """
    def __init__(self, embed_dim: int, num_heads: int, 
                 sr_ratio: int, ffn_ratio: int, drop_path: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = EfficientSelfAttention(embed_dim, num_heads, sr_ratio)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = MixFFN(embed_dim, ffn_ratio)
        
        # DropPath (Stochastic Depth): randomly drops entire residual branches
        # during training for regularization
        self.drop_path = nn.Identity() if drop_path == 0. else DropPath(drop_path)
    
    def forward(self, x: torch.Tensor, H: int, W: int) -> torch.Tensor:
        # Pre-norm + Efficient Attention + Residual
        x = x + self.drop_path(self.attn(self.norm1(x), H, W))
        # Pre-norm + Mix-FFN + Residual
        x = x + self.drop_path(self.ffn(self.norm2(x), H, W))
        return x


class DropPath(nn.Module):
    """Stochastic Depth - drops entire residual path with probability p."""
    def __init__(self, p: float = 0.1):
        super().__init__()
        self.p = p
    
    def forward(self, x):
        if not self.training or self.p == 0.:
            return x
        keep_prob = 1 - self.p
        # Random tensor per sample (not per element)
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = torch.bernoulli(torch.full(shape, keep_prob, device=x.device))
        return x * mask / keep_prob  # Scale to maintain expected value


print("\u2705 Core SegFormer components defined successfully")
print(f"   - OverlapPatchEmbedding: Tokenization without positional encoding")
print(f"   - EfficientSelfAttention: O(N²/R²) attention with Sequence Reduction")
print(f"   - MixFFN: FFN with depthwise conv for local + positional info")
print(f"   - TransformerBlock: Full block with pre-norm and residual")

In [0]:
# ============================================================================
# COMPONENT 5: Full MiT-B2 Encoder
# ============================================================================

class MiTB2Encoder(nn.Module):
    """
    Mix Transformer B2 Encoder - Hierarchical 4-stage encoder.
    
    Produces multi-scale features at 1/4, 1/8, 1/16, 1/32 of input resolution.
    Each stage: Patch Embed -> N x Transformer Blocks -> Output Feature Map
    
    B2 Configuration:
        embed_dims = [64, 128, 320, 512]
        num_heads = [1, 2, 5, 8]
        num_blocks = [2, 4, 4, 2]  (12 total blocks)
        sr_ratios = [8, 4, 2, 1]
        ffn_ratios = [8, 8, 4, 4]
    """
    def __init__(self, in_channels: int = 3, 
                 embed_dims: List[int] = [64, 128, 320, 512],
                 num_heads: List[int] = [1, 2, 5, 8],
                 num_blocks: List[int] = [2, 4, 4, 2],
                 sr_ratios: List[int] = [8, 4, 2, 1],
                 ffn_ratios: List[int] = [8, 8, 4, 4],
                 drop_path_rate: float = 0.1):
        super().__init__()
        self.num_stages = 4
        
        # Stochastic depth: linearly increasing drop rate across all blocks
        total_blocks = sum(num_blocks)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, total_blocks)]
        
        cur_block = 0
        for i in range(self.num_stages):
            # Patch embedding for this stage
            if i == 0:
                patch_embed = OverlapPatchEmbedding(
                    in_channels=in_channels, embed_dim=embed_dims[0],
                    patch_size=7, stride=4  # First stage: 4x downsample
                )
            else:
                patch_embed = OverlapPatchEmbedding(
                    in_channels=embed_dims[i-1], embed_dim=embed_dims[i],
                    patch_size=3, stride=2  # Subsequent stages: 2x downsample
                )
            
            # Transformer blocks for this stage
            blocks = nn.ModuleList([
                TransformerBlock(
                    embed_dim=embed_dims[i],
                    num_heads=num_heads[i],
                    sr_ratio=sr_ratios[i],
                    ffn_ratio=ffn_ratios[i],
                    drop_path=dpr[cur_block + j]
                )
                for j in range(num_blocks[i])
            ])
            cur_block += num_blocks[i]
            
            # Final LayerNorm for this stage
            norm = nn.LayerNorm(embed_dims[i])
            
            # Register as attributes
            setattr(self, f'patch_embed{i+1}', patch_embed)
            setattr(self, f'blocks{i+1}', blocks)
            setattr(self, f'norm{i+1}', norm)
    
    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        """
        Args:
            x: [B, 3, H, W] input image
        Returns:
            List of 4 feature maps: [F1, F2, F3, F4]
            F1: [B, 64, H/4, W/4]
            F2: [B, 128, H/8, W/8]
            F3: [B, 320, H/16, W/16]
            F4: [B, 512, H/32, W/32]
        """
        features = []
        
        for i in range(self.num_stages):
            # Get stage components
            patch_embed = getattr(self, f'patch_embed{i+1}')
            blocks = getattr(self, f'blocks{i+1}')
            norm = getattr(self, f'norm{i+1}')
            
            # Patch embedding: [B, C_in, H, W] -> [B, N, C]
            x, H, W = patch_embed(x)
            
            # Apply transformer blocks
            for block in blocks:
                x = block(x, H, W)
            
            # Final norm
            x = norm(x)
            
            # Reshape back to 2D: [B, N, C] -> [B, C, H, W]
            B, N, C = x.shape
            x = x.reshape(B, H, W, C).permute(0, 3, 1, 2)
            
            features.append(x)
        
        return features


# ============================================================================
# COMPONENT 6: All-MLP Decoder
# ============================================================================

class AllMLPDecoder(nn.Module):
    """
    All-MLP Decoder for SegFormer.
    
    Design philosophy: If the encoder is powerful enough (hierarchical + global),
    the decoder only needs to unify scales and project to classes.
    
    Pipeline per scale:
        F_i [C_i, H_i, W_i] -> Linear(C_i, C_d) -> Upsample(H/4, W/4)
    Then: Concat -> Linear(4*C_d, C_d) -> BN -> ReLU -> Dropout -> Linear(C_d, K)
    
    Args:
        encoder_dims: Channel dimensions from encoder stages [64, 128, 320, 512]
        decoder_dim: Unified decoder dimension C_d = 768 for B2
        num_classes: Number of segmentation classes (18 for clothes)
    """
    def __init__(self, encoder_dims: List[int] = [64, 128, 320, 512],
                 decoder_dim: int = 768, num_classes: int = 18,
                 dropout: float = 0.1):
        super().__init__()
        
        # Per-scale linear projections (implemented as 1x1 conv for spatial data)
        self.linear_layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(dim, decoder_dim, kernel_size=1),  # Channel projection
                nn.BatchNorm2d(decoder_dim),                 # Normalize
                nn.ReLU(inplace=True)
            )
            for dim in encoder_dims
        ])
        
        # Fusion: concatenated features -> unified representation
        self.linear_fuse = nn.Sequential(
            nn.Conv2d(decoder_dim * 4, decoder_dim, kernel_size=1),  # 4*C_d -> C_d
            nn.BatchNorm2d(decoder_dim),
            nn.ReLU(inplace=True)
        )
        
        # Classification head
        self.dropout = nn.Dropout2d(dropout)
        self.classifier = nn.Conv2d(decoder_dim, num_classes, kernel_size=1)
    
    def forward(self, features: List[torch.Tensor]) -> torch.Tensor:
        """
        Args:
            features: [F1, F2, F3, F4] from encoder
        Returns:
            logits: [B, num_classes, H/4, W/4] (1/4 resolution)
        """
        # Target spatial size: same as F1 (largest feature map = H/4 x W/4)
        target_size = features[0].shape[2:]  # (H/4, W/4)
        
        projected = []
        for i, (feat, linear) in enumerate(zip(features, self.linear_layers)):
            # Linear project: [B, C_i, H_i, W_i] -> [B, C_d, H_i, W_i]
            x = linear(feat)
            
            # Upsample to target size (bilinear interpolation)
            # F1: no upsample (already target size)
            # F2: 2x, F3: 4x, F4: 8x
            if x.shape[2:] != target_size:
                x = F.interpolate(x, size=target_size, 
                                  mode='bilinear', align_corners=False)
            projected.append(x)
        
        # Concatenate all scales: [B, 4*C_d, H/4, W/4]
        fused = torch.cat(projected, dim=1)
        
        # Fuse: [B, 4*C_d, H/4, W/4] -> [B, C_d, H/4, W/4]
        fused = self.linear_fuse(fused)
        
        # Classify: [B, C_d, H/4, W/4] -> [B, K, H/4, W/4]
        logits = self.classifier(self.dropout(fused))
        
        return logits


# ============================================================================
# COMPONENT 7: Complete SegFormer Model
# ============================================================================

class SegFormerB2Clothes(nn.Module):
    """
    Complete SegFormer-B2 for Clothes Segmentation.
    
    Encoder: MiT-B2 (hierarchical transformer, ~24.7M params)
    Decoder: All-MLP (lightweight, ~2.7M params)
    Total: ~27.4M parameters
    
    Input: [B, 3, H, W] RGB image (any resolution, typically 512x512 or 1024x1024)
    Output: [B, 18, H, W] per-pixel logits for 18 clothing classes
    """
    def __init__(self, num_classes: int = 18):
        super().__init__()
        self.encoder = MiTB2Encoder()
        self.decoder = AllMLPDecoder(
            encoder_dims=[64, 128, 320, 512],
            decoder_dim=768,
            num_classes=num_classes
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Full forward pass.
        
        Args:
            x: [B, 3, H, W] input image
        Returns:
            [B, num_classes, H, W] per-pixel logits at original resolution
        """
        input_size = x.shape[2:]  # (H, W)
        
        # Encoder: extract multi-scale features
        features = self.encoder(x)  # [F1, F2, F3, F4]
        
        # Decoder: fuse and classify at 1/4 resolution
        logits = self.decoder(features)  # [B, K, H/4, W/4]
        
        # Upsample to original resolution
        logits = F.interpolate(logits, size=input_size, 
                               mode='bilinear', align_corners=False)
        
        return logits  # [B, K, H, W]


# ============================================================================
# Verify model construction and parameter count
# ============================================================================

model = SegFormerB2Clothes(num_classes=18)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
encoder_params = sum(p.numel() for p in model.encoder.parameters())
decoder_params = sum(p.numel() for p in model.decoder.parameters())

print(f"\u2705 SegFormer-B2 Clothes model constructed successfully")
print(f"   Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"   Encoder parameters: {encoder_params:,} ({encoder_params/1e6:.1f}M)")
print(f"   Decoder parameters: {decoder_params:,} ({decoder_params/1e6:.1f}M)")

# Test forward pass
with torch.no_grad():
    dummy_input = torch.randn(1, 3, 512, 512)
    output = model(dummy_input)
    print(f"\n   Input shape: {dummy_input.shape}")
    print(f"   Output shape: {output.shape}")
    print(f"   Output represents: per-pixel logits for 18 clothing classes")

In [0]:
# ============================================================================
# USING THE ACTUAL PRETRAINED MODEL FROM HUGGING FACE
# mattmdjaga/segformer_b2_clothes
# ============================================================================
# This demonstrates the production inference pipeline using HuggingFace
# transformers library - the recommended way to use this model.

from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
from PIL import Image
import numpy as np
import torch
import torch.nn.functional as F
import requests
from io import BytesIO

# ---- Step 1: Load processor and model ----
# The processor handles:
#   - Resize to model's expected input (512x512)
#   - Normalize with ImageNet mean/std
#   - Convert to tensor
# The model is SegFormer-B2 fine-tuned on clothing segmentation dataset

processor = SegformerImageProcessor.from_pretrained("mattmdjaga/segformer_b2_clothes")
model_hf = SegformerForSemanticSegmentation.from_pretrained("mattmdjaga/segformer_b2_clothes")

# Move to evaluation mode (disables dropout, batchnorm in eval mode)
model_hf.eval()

print("Model loaded successfully!")
print(f"Model type: {model_hf.config.model_type}")
print(f"Number of labels: {model_hf.config.num_labels}")
print(f"Image size expected: {processor.size}")
print(f"\nLabel mapping:")
for idx, label in model_hf.config.id2label.items():
    print(f"  {idx}: {label}")

In [0]:
# ============================================================================
# FULL INFERENCE PIPELINE: Image -> Segmentation Map
# ============================================================================

# ---- Step 2: Load a sample image ----
url = "https://plus.unsplash.com/premium_photo-1673210886161-bfcc40f54d1f?w=600"
try:
    response = requests.get(url, timeout=10)
    image = Image.open(BytesIO(response.content)).convert("RGB")
    print(f"Image loaded: {image.size} (W x H)")
except:
    # Fallback: create a synthetic test image
    image = Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))
    print(f"Using synthetic test image: {image.size}")

# ---- Step 3: Preprocess ----
# processor.preprocess() does:
#   1. Resize to (512, 512) using bilinear interpolation
#   2. Rescale pixel values from [0, 255] to [0, 1] (divide by 255)
#   3. Normalize: (pixel - mean) / std
#      mean = [0.485, 0.456, 0.406] (ImageNet)
#      std  = [0.229, 0.224, 0.225] (ImageNet)
#   4. Convert to PyTorch tensor [1, 3, 512, 512]

inputs = processor(images=image, return_tensors="pt")
print(f"\nPreprocessed input shape: {inputs['pixel_values'].shape}")
print(f"Pixel value range: [{inputs['pixel_values'].min():.3f}, {inputs['pixel_values'].max():.3f}]")

# ---- Step 4: Forward pass (inference) ----
with torch.no_grad():  # No gradient computation needed for inference
    outputs = model_hf(**inputs)

# outputs.logits shape: [B, num_classes, H/4, W/4] = [1, 18, 128, 128]
# Note: model outputs at 1/4 resolution to save memory
print(f"\nRaw output logits shape: {outputs.logits.shape}")
print(f"  Batch size: {outputs.logits.shape[0]}")
print(f"  Num classes: {outputs.logits.shape[1]}")
print(f"  Spatial size: {outputs.logits.shape[2]}x{outputs.logits.shape[3]} (1/4 of input)")

# ---- Step 5: Post-processing ----
# Upsample logits to original image size
upsampled_logits = F.interpolate(
    outputs.logits,
    size=image.size[::-1],  # (H, W) - PIL gives (W, H) so reverse
    mode='bilinear',
    align_corners=False
)
print(f"\nUpsampled logits shape: {upsampled_logits.shape}")

# Convert logits to class predictions via argmax
# For each pixel, select the class with highest logit value
predicted_mask = upsampled_logits.argmax(dim=1)  # [B, H, W]
print(f"Prediction mask shape: {predicted_mask.shape}")
print(f"Unique classes predicted: {torch.unique(predicted_mask).tolist()}")

# ---- Step 6: Map class IDs to names ----
id2label = model_hf.config.id2label
print(f"\nClasses found in this image:")
for class_id in torch.unique(predicted_mask).tolist():
    pixel_count = (predicted_mask == class_id).sum().item()
    total_pixels = predicted_mask.numel()
    percentage = 100.0 * pixel_count / total_pixels
    print(f"  {id2label[class_id]:15s}: {pixel_count:>7,} pixels ({percentage:.1f}%)")

## 4. Complexity Analysis

### 4.1 Time Complexity

#### Per-Stage Attention Complexity

| Stage | Tokens $$N$$ | SR Ratio $$R$$ | Attention Complexity | Standard Attention |
|-------|-----------|------|---------------------|--------------------|
| 1 | 16,384 | 8 | $$\mathcal{O}(16384 \times 256 \times 64) = \mathcal{O}(268M)$$ | $$\mathcal{O}(16384^2 \times 64) = \mathcal{O}(17.2B)$$ |
| 2 | 4,096 | 4 | $$\mathcal{O}(4096 \times 256 \times 128) = \mathcal{O}(134M)$$ | $$\mathcal{O}(4096^2 \times 128) = \mathcal{O}(2.1B)$$ |
| 3 | 1,024 | 2 | $$\mathcal{O}(1024 \times 256 \times 320) = \mathcal{O}(84M)$$ | $$\mathcal{O}(1024^2 \times 320) = \mathcal{O}(336M)$$ |
| 4 | 256 | 1 | $$\mathcal{O}(256^2 \times 512) = \mathcal{O}(33.5M)$$ | Same (no reduction) |

**Total reduction factor**: ~64x at stage 1, ~16x at stage 2, ~4x at stage 3.

#### Overall Forward Pass Complexity

For input size $$H \times W$$:
$$T_{\text{total}} = \sum_{i=1}^{4} L_i \cdot \left[\mathcal{O}\left(\frac{N_i^2}{R_i^2} \cdot C_i\right) + \mathcal{O}(N_i \cdot C_i \cdot E_i \cdot C_i)\right]$$

where $$L_i$$ = num blocks, $$E_i$$ = FFN expansion ratio.

### 4.2 Space Complexity

| Component | Memory (fp32, B=1, 512×512) |
|-----------|-----------------------------|
| Input image | 3 MB |
| Stage 1 activations (F1) | 64 × 128 × 128 × 4B = 4.2 MB |
| Stage 2 activations (F2) | 128 × 64 × 64 × 4B = 2.1 MB |
| Stage 3 activations (F3) | 320 × 32 × 32 × 4B = 1.3 MB |
| Stage 4 activations (F4) | 512 × 16 × 16 × 4B = 0.5 MB |
| Attention maps (peak, Stage 1) | 16384 × 256 × 4B = 16.8 MB |
| Decoder features | 768 × 128 × 128 × 4B = 50.3 MB |
| **Model parameters** | **27.4M × 4B = 109.6 MB** |
| **Peak activation memory** | **~200 MB** |
| **Total inference** | **~310 MB** |
| **Training (with gradients)** | **~900 MB - 1.2 GB** |

### 4.3 Inference Latency

| Resolution | FLOPs (GMACs) | Latency (GPU V100) | Latency (CPU) |
|-----------|---------------|--------------------|--------------|
| 512 × 512 | ~62 GMACs | ~15 ms | ~800 ms |
| 1024 × 1024 | ~248 GMACs | ~55 ms | ~3200 ms |

### 4.4 Scaling Behavior

- **Spatial resolution**: $$\mathcal{O}(H \cdot W)$$ for most operations (linear in pixel count) due to SR
- **Batch size**: Linear scaling in both memory and compute
- **Number of classes**: Negligible impact (only final 1×1 conv changes)
- **Sequence length vs. standard ViT**: SegFormer scales as $$\mathcal{O}(N)$$ vs ViT's $$\mathcal{O}(N^2)$$ for attention

---

## 5. Training Details

### 5.1 Pre-training (ImageNet-1K, Encoder Only)

| Hyperparameter | Value | Rationale |
|---------------|-------|-----------|
| Optimizer | AdamW | Decoupled weight decay; better than Adam for transformers |
| Learning rate | $$6 \times 10^{-5}$$ (B2) | Lower than CNNs; transformers are sensitive to LR |
| LR Schedule | Polynomial decay (power=1.0) | Smooth decay, avoids sudden drops |
| Weight decay | 0.01 | Regularization; AdamW decouples from LR |
| Warmup | 1500 iterations | Stabilizes early training with random weights |
| Batch size | 16 per GPU | Memory-constrained due to high-res features |
| Training iterations | 160K | Standard for segmentation fine-tuning |
| Augmentation | RandomResize(0.5-2.0), RandomCrop(512), RandomFlip | Standard seg augmentation |
| Dropout | 0.0 (encoder), 0.1 (decoder) | DropPath provides sufficient regularization |
| DropPath rate | 0.1 (stochastic depth) | Linearly increases across blocks |

### 5.2 Fine-tuning for Clothes Segmentation

| Hyperparameter | Value |
|---------------|-------|
| Base model | nvidia/mit-b2 (ImageNet-1K pretrained) |
| Dataset | ATR (human parsing dataset) / similar clothing dataset |
| Num classes | 18 |
| Loss | Cross-Entropy (pixel-wise) |
| Optimizer | AdamW |
| Learning rate | ~$$10^{-4}$$ to $$6 \times 10^{-5}$$ (typical for fine-tuning) |
| Input resolution | 512 × 512 |
| Epochs | ~50-100 (depending on dataset size) |

### 5.3 Optimizer: AdamW

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
$$\theta_t = \theta_{t-1} - \eta\left(\frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda\theta_{t-1}\right)$$

where $$\beta_1 = 0.9, \beta_2 = 0.999, \epsilon = 10^{-8}, \lambda = 0.01$$ (weight decay).

**Why AdamW over Adam**: In Adam, weight decay is coupled with the adaptive learning rate, making its effect inconsistent across parameters. AdamW decouples weight decay, applying it directly to parameters regardless of gradient magnitude.

### 5.4 Learning Rate Schedule: Polynomial Decay

$$\eta_t = \eta_{\text{base}} \cdot \left(1 - \frac{t}{T_{\text{max}}}\right)^{\text{power}}$$

With power = 1.0 (linear decay). Provides gradual reduction without the sharp drops of step schedules.

### 5.5 Data Augmentation Strategy

1. **Random Resize**: Scale factor in [0.5, 2.0] → multi-scale training
2. **Random Crop**: 512 × 512 from resized image → fixed input size
3. **Random Horizontal Flip**: p=0.5 → left-right symmetry
4. **PhotoMetric Distortion**: Color jitter for robustness
5. **Normalization**: ImageNet mean/std

### 5.6 Gradient Clipping

Max gradient norm = 1.0 (typical for transformer training):
$$\hat{g} = \frac{g}{\max(1, \|g\|_2 / \text{max\_norm})}$$

---

## 6. Optimizations

### 6.1 Computational Optimizations

| Optimization | Technique | Speedup |
|-------------|-----------|--------|
| Sequence Reduction | Strided conv on K,V | 4-64x per attention layer |
| Depthwise Conv | Groups=channels in Mix-FFN | 9x fewer params vs standard conv |
| Lightweight decoder | MLP vs ASPP/FPN | 3-5x decoder speedup |
| 1/4 output resolution | Upsample only at end | 16x fewer decoder operations |

### 6.2 Memory Optimizations

| Technique | Description | Saving |
|-----------|-------------|--------|
| Mixed Precision (FP16/BF16) | Half-precision for activations & gradients | ~50% memory |
| Gradient Checkpointing | Recompute activations in backward pass | ~60% activation memory |
| Fused operations | Combined LayerNorm + Linear | ~10% speedup |
| In-place operations | ReLU(inplace=True) | Minimal memory saving |

### 6.3 Quantization

```
FP32 (baseline):     27.4M params × 4 bytes = 109.6 MB
FP16 (half):         27.4M params × 2 bytes = 54.8 MB  (minimal quality loss)
INT8 (quantized):    27.4M params × 1 byte  = 27.4 MB  (~1-2% mIoU drop)
INT4 (aggressive):   27.4M params × 0.5 byte = 13.7 MB (noticeable quality loss)
```

### 6.4 ONNX/TensorRT Deployment

1. Export to ONNX: `torch.onnx.export(model, dummy_input, "segformer_b2.onnx")`
2. Optimize with TensorRT: fuse layers, optimize memory, select precision
3. Typical speedup: 2-4x over PyTorch on NVIDIA GPUs

### 6.5 Batch Inference Optimization

- Pad images to same size within batch (avoid dynamic shapes)
- Use `torch.cuda.amp.autocast()` for automatic mixed precision
- Pin memory for CPU-to-GPU transfer: `DataLoader(pin_memory=True)`

---

## 7. Practical Implementation Guide

### 7.1 Dataset Preparation

```python
# Expected format:
# images/: RGB images (any format: jpg, png)
# masks/:  Single-channel label maps (values 0-17 for 18 classes)
# Each pixel in mask = class ID (0=background, 1=hat, ... 17=scarf)
```

### 7.2 Common Failure Cases

1. **Occluded clothing**: Partially visible items may be missed
2. **Unusual poses**: Extreme poses reduce accuracy
3. **Multiple people**: Model segments all visible clothing (no instance separation)
4. **Ambiguous boundaries**: Where upper-clothes meets pants, boundary may be imprecise
5. **Rare items**: Scarves, belts have fewer training samples → lower accuracy
6. **Low resolution**: Small people in large images need pre-cropping

### 7.3 Production Deployment Checklist

- [ ] Convert to ONNX/TorchScript for framework-independent deployment
- [ ] Apply FP16 quantization for 2x memory reduction
- [ ] Implement batched inference for throughput
- [ ] Add input validation (image size, format, channels)
- [ ] Implement confidence thresholding (softmax > threshold)
- [ ] Add fallback for out-of-distribution inputs
- [ ] Monitor inference latency and memory usage
- [ ] Version model artifacts with checksums

## 8. Fully Worked Numerical Example

### 8.1 Setup: Tiny Input Through Stage 1 (Simplified)

Let's trace a **4×4 pixel, 3-channel** input through a simplified Stage 1 with:
- Patch embedding: kernel=3, stride=2, padding=1 → output 2×2 = 4 tokens
- Embed dim C = 4, 1 head, SR ratio = 2

#### Step 1: Input
```
X_input = [B=1, C=3, H=4, W=4]  (48 values)
Simplified: just track one 4×4 channel
Channel 0: [[0.1, 0.2, 0.3, 0.4],
            [0.5, 0.6, 0.7, 0.8],
            [0.9, 1.0, 1.1, 1.2],
            [1.3, 1.4, 1.5, 1.6]]
```

#### Step 2: Overlapping Patch Embedding (Conv2d k=3, s=2, p=1)

Output size: $$\lfloor(4 + 2 - 3)/2\rfloor + 1 = 2$$

So output: [1, 4, 2, 2] → reshape → [1, 4, 4] (4 tokens, dim 4)

```
After conv + flatten + LN:
X_tokens = [[t1_0, t1_1, t1_2, t1_3],   # Token 1 (top-left patch)
            [t2_0, t2_1, t2_2, t2_3],   # Token 2 (top-right patch)
            [t3_0, t3_1, t3_2, t3_3],   # Token 3 (bottom-left patch)
            [t4_0, t4_1, t4_2, t4_3]]   # Token 4 (bottom-right patch)

Suppose after conv+LN:
X = [[0.5, -0.3, 0.8, -0.1],
     [0.2,  0.7, -0.4, 0.6],
     [-0.1, 0.4, 0.9, -0.5],
     [0.3, -0.2, 0.1,  0.8]]
```
Shape: [1, 4, 4] = [B, N=4, C=4]

#### Step 3: Layer Normalization

For token 1: $$\mathbf{x} = [0.5, -0.3, 0.8, -0.1]$$

$$\mu = (0.5 - 0.3 + 0.8 - 0.1)/4 = 0.225$$

$$\sigma^2 = [(0.5-0.225)^2 + (-0.3-0.225)^2 + (0.8-0.225)^2 + (-0.1-0.225)^2]/4$$
$$= [0.0756 + 0.2756 + 0.3306 + 0.1056]/4 = 0.1969$$

$$\sigma = 0.4437$$

Normalized (assuming $$\gamma=1, \beta=0$$):
$$\hat{x}_1 = (0.5 - 0.225)/0.4437 = 0.620$$
$$\hat{x}_2 = (-0.3 - 0.225)/0.4437 = -1.183$$
$$\hat{x}_3 = (0.8 - 0.225)/0.4437 = 1.296$$
$$\hat{x}_4 = (-0.1 - 0.225)/0.4437 = -0.733$$

#### Step 4: Efficient Self-Attention (SR=2)

**Q projection**: $$Q = X_{norm} \cdot W_Q$$ where $$W_Q \in \mathbb{R}^{4 \times 4}$$
```
Q shape: [1, 4, 4] (4 tokens, all serve as queries)
```

**Sequence Reduction for K, V** (SR ratio = 2):
1. Reshape X to 2D: [1, 4, 2, 2] (C=4, H=2, W=2)
2. Apply Conv2d(4, 4, k=2, s=2): [1, 4, 1, 1] (reduces 2×2 → 1×1)
3. Flatten: [1, 1, 4] → **only 1 key-value token!**
4. Apply K,V projections: K=[1, 1, 4], V=[1, 1, 4]

**Attention computation**:
$$\text{attn} = \text{softmax}(Q \cdot K^T / \sqrt{4})$$

$$Q \cdot K^T$$: [1, 4, 4] × [1, 4, 1] = [1, 4, 1]

Since there's only 1 key, softmax over length-1 gives all 1.0:
```
attn_weights = [[1.0],    # Every query attends fully to the single key
                [1.0],
                [1.0],
                [1.0]]
```

Output = attn × V = all tokens get the same value vector (this is the extreme case with SR=2 on tiny input).

#### Step 5: Mix-FFN

```
Input: [1, 4, 4]  (4 tokens, C=4)
  → Linear(4, 16): [1, 4, 16]  (expansion ratio=4)
  → Reshape to 2D: [1, 16, 2, 2]
  → DWConv3×3(16, 16, groups=16): [1, 16, 2, 2]
  → GELU activation
  → Reshape to seq: [1, 4, 16]
  → Linear(16, 4): [1, 4, 4]
```

**GELU on example value 0.620**:
$$\text{GELU}(0.620) = 0.620 \times \Phi(0.620) = 0.620 \times 0.7324 = 0.454$$

#### Step 6: Residual Connections
```
x_after_attn = x + attention_output     # [1, 4, 4]
x_after_ffn  = x_after_attn + ffn_output # [1, 4, 4]
```

#### Step 7: Reshape to Feature Map
```
Output F1 = [1, 4, 2, 2]  (C=4, H=2, W=2)
```

### 8.2 Decoder Numerical Example (Simplified)

Given features at 4 scales (simplified to 2D for clarity):
```
F1: [1, 4, 2, 2]  (1/4 res, C=4)
F2: [1, 8, 1, 1]  (1/8 res, C=8)  
F3: [1, 16, 1, 1] (1/16 res, C=16) -- would be smaller but using 1×1 for simplicity
F4: [1, 32, 1, 1] (1/32 res, C=32)

Decoder dim C_d = 8
```

**Step 1: Linear project each scale to C_d=8**
```
F1: Conv1×1(4→8): [1, 8, 2, 2]
F2: Conv1×1(8→8): [1, 8, 1, 1]
F3: Conv1×1(16→8): [1, 8, 1, 1]
F4: Conv1×1(32→8): [1, 8, 1, 1]
```

**Step 2: Upsample all to F1's resolution (2×2)**
```
F1: [1, 8, 2, 2] (already correct)
F2: bilinear 1×1→2×2: [1, 8, 2, 2]
F3: bilinear 1×1→2×2: [1, 8, 2, 2]
F4: bilinear 1×1→2×2: [1, 8, 2, 2]
```

**Step 3: Concatenate**
```
Concat: [1, 32, 2, 2]  (4 × C_d = 4 × 8 = 32)
```

**Step 4: Fuse**
```
Conv1×1(32→8): [1, 8, 2, 2]
```

**Step 5: Classify**
```
Conv1×1(8→18): [1, 18, 2, 2]  → 18 logits per pixel
```

**Step 6: Argmax for prediction**
```
For pixel (0,0), logits = [0.1, -0.5, 0.3, 0.8, 2.1, ...] (18 values)
argmax = 4 → Class "Upper-clothes"
```

### 8.3 Loss Computation Example

For one pixel with ground truth class = 4 (Upper-clothes) and logits:
$$z = [0.1, -0.5, 0.3, 0.8, 2.1, -0.3, 0.4, -0.1, 0.0, -0.2, 0.1, 0.5, -0.4, 0.2, 0.3, -0.1, 0.0, -0.3]$$

**Log-Sum-Exp** (with max trick, $$m = 2.1$$):
$$\text{LSE} = 2.1 + \log(e^{-2.0} + e^{-2.6} + e^{-1.8} + e^{-1.3} + e^{0} + \ldots)$$
$$= 2.1 + \log(0.135 + 0.074 + 0.165 + 0.273 + 1.0 + 0.074 + 0.182 + \ldots)$$
$$\approx 2.1 + \log(3.08) \approx 2.1 + 1.12 = 3.22$$

**Cross-entropy loss**:
$$\ell = -z_4 + \text{LSE} = -2.1 + 3.22 = 1.12$$

**Softmax probability for correct class**:
$$p_4 = e^{2.1} / e^{3.22} = e^{-1.12} \approx 0.326$$

The model is ~32.6% confident in the correct class → loss is moderate.

## 9. Intuition: Why Each Component Works

### 9.1 Component Contribution Analysis

| Component | What it contributes | What happens if removed |
|-----------|--------------------|--------------------------|
| Hierarchical Encoder | Multi-scale features (local + global) | Single-scale = poor on small objects OR poor global context |
| Sequence Reduction | Tractable attention on high-res features | Quadratic memory/compute blow-up at stages 1-2 |
| Overlapping Patch Embed | Position info without PE; smooth tokenization | Must add learnable PE; resolution inflexibility; boundary artifacts |
| Mix-FFN (DWConv) | Local spatial inductive bias + position info | ~0.5% mIoU drop; less robust to resolution changes |
| All-MLP Decoder | Simplicity; proves encoder sufficiency | (replaced with heavy decoder) marginal gain, 3-5x slower |
| DropPath | Regularization via stochastic depth | Overfitting on smaller datasets |
| LayerNorm (Pre-Norm) | Training stability | Divergence with Post-Norm at depth |

### 9.2 Common Misconceptions

1. **"SegFormer uses no convolutions"** → FALSE. It uses convolutions extensively: patch embedding (strided conv), SR (strided conv), Mix-FFN (depthwise conv), decoder (1×1 conv)
2. **"Attention is the main computational cost"** → FALSE for SegFormer. Due to SR, the FFN layers dominate FLOPs
3. **"No positional encoding means no position awareness"** → FALSE. Position is encoded through overlapping patches, DWConv zero-padding, and hierarchical structure
4. **"Lightweight decoder = worse performance"** → FALSE. The All-MLP decoder matches or exceeds heavy decoders because the encoder features are already rich enough

---

## 10. Detailed Comparisons

### 10.1 SegFormer Variants

| Model | Encoder Params | Total Params | ADE20K mIoU | Cityscapes mIoU | FLOPs (512×512) |
|-------|---------------|-------------|-------------|-----------------|------------------|
| B0 | 3.4M | 3.8M | 37.4% | 76.2% | 8.4G |
| B1 | 13.2M | 13.7M | 42.2% | 78.5% | 15.9G |
| **B2** | **24.7M** | **27.4M** | **46.5%** | **81.0%** | **62.4G** |
| B3 | 44.6M | 47.3M | 48.5% | 81.7% | 79.0G |
| B4 | 60.8M | 64.1M | 49.6% | 82.3% | 95.7G |
| B5 | 81.4M | 84.7M | 50.0% | 82.4% | 183.3G |

### 10.2 SegFormer vs. Other Architectures (ADE20K)

| Method | Backbone | Params | mIoU (SS) | FLOPs | Speed (fps) |
|--------|----------|--------|-----------|-------|-------------|
| DeepLabV3+ | ResNet-101 | 63M | 44.1% | 255G | 15 |
| SETR-PUP | ViT-Large | 318M | 48.6% | 1240G | 3 |
| Swin-T + UPerNet | Swin-T | 60M | 44.5% | 236G | 14 |
| **SegFormer-B2** | **MiT-B2** | **27.4M** | **46.5%** | **62.4G** | **30** |
| SegFormer-B5 | MiT-B5 | 84.7M | 50.0% | 183G | 11 |

**Key insight**: SegFormer-B2 achieves comparable mIoU to methods with 2-10x more parameters and FLOPs.

---

## 11. Interview Questions (Beginner to Research Level)

### Beginner Level

**Q1: What is semantic segmentation?**
> A: Assigning a class label to every pixel in an image. Unlike classification (one label per image) or detection (bounding boxes), segmentation provides pixel-precise boundaries.

**Q2: Why does SegFormer use overlapping patches instead of non-overlapping?**
> A: Overlapping patches (stride < kernel) share boundary information between adjacent tokens, implicitly encoding spatial position. This eliminates the need for explicit positional encodings, making the model resolution-agnostic at inference.

**Q3: What are the 18 classes in the clothes segmentation model?**
> A: Background, Hat, Hair, Sunglasses, Upper-clothes, Skirt, Pants, Dress, Belt, Left-shoe, Right-shoe, Face, Left-leg, Right-leg, Left-arm, Right-arm, Bag, Scarf.

### Intermediate Level

**Q4: Explain Sequence Reduction (SR) and its complexity benefit.**
> A: SR reduces the key/value sequence from $$N$$ to $$N/R^2$$ tokens by reshaping the spatial feature map into $$R \times R$$ groups and projecting each group to a single token via a linear layer. This reduces attention complexity from $$\mathcal{O}(N^2 \cdot C)$$ to $$\mathcal{O}(N^2/R^2 \cdot C)$$. The reduction is stage-dependent: aggressive in early stages (large spatial size) and minimal/none in later stages (small spatial size).

**Q5: Why does SegFormer use Pre-Norm instead of Post-Norm?**
> A: Pre-Norm (LayerNorm before attention/FFN) provides more stable gradients during training, especially for deep networks. With Post-Norm, the residual connection adds unnormalized attention output to the input, which can cause exploding activations. Pre-Norm ensures the input to attention/FFN is always well-scaled.

**Q6: How does the All-MLP decoder work and why is it sufficient?**
> A: For each of 4 encoder scales, it applies a linear projection to a common dimension ($$C_d=768$$), upsamples to 1/4 resolution, concatenates all scales, fuses with another linear layer, and classifies. It works because the hierarchical encoder already captures both local details (early stages) and global context (later stages) — the decoder just needs to unify and project.

### Advanced Level

**Q7: Compare the information loss between SegFormer's SR and PVT's SRA (Spatial Reduction Attention).**
> A: SegFormer uses a Conv2d for SR (learned spatial grouping with local receptive field), while PVT uses reshape + linear (which loses spatial locality during the linear projection). SegFormer's conv-based SR preserves local spatial structure within each reduced token, leading to better boundary preservation. However, both lose fine-grained token-level detail that full attention preserves.

**Q8: How does SegFormer handle variable resolution at inference without positional encoding?**
> A: Since there's no positional encoding to interpolate, the model can directly process any resolution. The Conv2d operations (patch embed, DWConv, SR) naturally adapt to different spatial sizes. The only resolution-dependent behavior is the decoder's bilinear upsampling, which is parametric-free. This makes SegFormer ideal for deployment where input resolution varies.

**Q9: Analyze the trade-off between SR ratio and performance. When would you change the default ratios?**
> A: Higher SR ratios (more aggressive reduction) trade fine-grained attention for efficiency. For tasks requiring precise boundaries (medical imaging), you might reduce SR ratios at early stages (e.g., [4,2,1,1] instead of [8,4,2,1]) at the cost of 4x more attention compute. For very high-resolution inputs (4K images), you might increase SR ratios ([16,8,4,2]) to keep inference tractable.

### Research Level

**Q10: How would you modify SegFormer for instance segmentation?**
> A: Options: (a) Add a Mask2Former-style mask decoder that uses the SegFormer encoder features as the pixel decoder, with learned object queries predicting per-instance masks. (b) Add a center-prediction branch (like PanopticFPN) to distinguish instances of the same class. (c) Use SegFormer features as input to a SOLO/CondInst-style dynamic convolution head. The key challenge is that the All-MLP decoder doesn't produce per-instance features natively.

**Q11: The Mix-FFN uses 3×3 depthwise convolution. What would happen with larger kernels (5×5, 7×7)?**
> A: Larger kernels would provide larger local receptive fields in the FFN, potentially reducing the burden on attention for capturing local structure. However: (a) 5×5 DWConv adds 2.8x params vs 3×3, (b) the attention mechanism already handles beyond-local relationships, making larger FFN kernels redundant, (c) empirically, 3×3 provides the best efficiency/performance trade-off. Recent works (ConvNeXt) show that 7×7 depthwise convs can be effective, but in SegFormer's hybrid design, the attention handles the global mixing role.

---

## 12. Common Mistakes

### 12.1 Implementation Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Forgetting to upsample logits to original size | Predictions at 1/4 resolution | Add `F.interpolate(logits, size=original_size)` |
| Using wrong normalization (BatchNorm in encoder) | Performance degradation | Encoder uses LayerNorm; decoder can use BN |
| Not using `align_corners=False` in interpolation | Slight misalignment at boundaries | Always set `align_corners=False` for bilinear |
| Applying softmax before loss | NaN/incorrect gradients | `CrossEntropyLoss` expects raw logits |
| Wrong image normalization (not ImageNet stats) | Distribution shift from pretraining | Use mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225] |

### 12.2 Training Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Learning rate too high for fine-tuning | Divergence or catastrophic forgetting | Use 1/10th of pre-training LR for backbone |
| No multi-scale training | Poor generalization to different object sizes | RandomResize(0.5, 2.0) |
| Ignoring class imbalance | Rare classes (belt, scarf) underperform | Class-weighted loss or focal loss |
| Training too long without validation | Overfitting | Monitor val mIoU, use early stopping |

### 12.3 Debugging Strategies

1. **All predictions same class**: Check loss computation, learning rate, data loading
2. **Noisy boundaries**: Increase resolution, reduce SR ratio, add CRF post-processing
3. **NaN loss**: Reduce LR, check for empty masks, add epsilon to log operations
4. **Memory OOM**: Reduce batch size, use gradient checkpointing, reduce input size
5. **Poor transfer**: Use differential learning rates (lower for encoder, higher for decoder)

---

## 13. Summary: Key Takeaways

### Core Architecture Formula
$$\text{SegFormer} = \underbrace{\text{MiT Encoder}}_{\text{Hierarchical Transformer}} + \underbrace{\text{All-MLP Decoder}}_{\text{Lightweight Fusion}}$$

### Essential Equations
1. **Efficient Attention**: $$\text{Attn}(Q, \hat{K}, \hat{V}) = \text{softmax}(Q\hat{K}^T/\sqrt{d_h})\hat{V}$$ where $$\hat{K} \in \mathbb{R}^{N/R^2 \times C}$$
2. **Mix-FFN**: $$\text{GELU}(\text{DWConv}_{3\times3}(xW_1))W_2$$
3. **Loss**: $$\mathcal{L} = -\frac{1}{|\Omega|}\sum_{(i,j)} \log \text{softmax}(z_{ij})_{y_{ij}}$$

### Key Design Decisions
- **No positional encoding** → Overlap patches + DWConv provide position implicitly
- **Hierarchical stages** → Multi-scale without FPN complexity
- **Stage-dependent SR** → [8,4,2,1] balances efficiency and detail preservation
- **Pre-Norm residuals** → Stable deep training
- **Simple decoder** → Proves encoder quality; fast inference

### Practical Recommendations
1. Use `mattmdjaga/segformer_b2_clothes` via HuggingFace for production clothes parsing
2. For custom domains: fine-tune from `nvidia/mit-b2`, use AdamW with poly LR schedule
3. For speed-critical apps: Use B0/B1; for accuracy-critical: Use B4/B5
4. Always use FP16 inference for 2x memory savings with negligible quality impact
5. Post-process with connected components to remove small noise regions

### Model Card Quick Reference
```
Model: mattmdjaga/segformer_b2_clothes
Architecture: SegFormer-B2 (MiT-B2 encoder + All-MLP decoder)
Params: ~27.4M
Input: RGB image (any resolution, recommended 512×512+)
Output: 18-class segmentation map
Pretrained: ImageNet-1K (encoder) + Clothing dataset (full model)
Framework: HuggingFace Transformers
License: Refer to model card
```

---

*Note: Specific fine-tuning hyperparameters for `mattmdjaga/segformer_b2_clothes` (exact dataset, epochs, LR) are not fully documented on the model card. The training details in Section 5 are based on the original SegFormer paper (Xie et al., NeurIPS 2021) and standard practices for this architecture.*